# 01d · Extract DFDC face crops (held-out cross-dataset test)

**DFDC** (Deepfake Detection Challenge, Facebook/Meta 2020) is the **cross-dataset test** for v2: different
actors, cameras, lighting and face-swap methods from everything the models are trained on. It is **test only**:
never used for training, early stopping or calibration, so its numbers measure real generalisation.

Labels come from each folder's `metadata.json`. DFDC is about 80% fake, so by default every real video is kept
and the same number of fake videos is sampled (seeded).

**Notebook settings (right sidebar)**
- **Accelerator:** None (CPU) · **Internet:** On
- **Inputs:** DFDC videos with `metadata.json`: the competition data (`deepfake-detection-challenge`,
  `train_sample_videos/` = 400 videos) and/or any `dfdc_train_part_N` dataset (more parts = more real videos)

**Produces:** `/kaggle/working/data` (DFDC test crops + `manifest.csv`). Add it as an input to **03**.

In [ ]:
# --- Project code -------------------------------------------------------------------------
# Always clones the latest code. It lives in /tmp, OUTSIDE /kaggle/working, so it is never saved into this
# notebook's output (otherwise later notebooks would pick up a stale copy). For a private repo add a
# GITHUB_TOKEN secret. Fallback: a zip of the project uploaded as a Kaggle *dataset*.
import os, sys, glob, shutil, subprocess

PROJECT = "/tmp/dfdc-project"
REPO, BRANCH = "https://github.com/Amay-M-Nair/DFDC.git", "dev"

if not os.path.exists(PROJECT):
    token = ""
    try:
        from kaggle_secrets import UserSecretsClient
        token = UserSecretsClient().get_secret("GITHUB_TOKEN")
    except Exception:
        pass
    url = REPO.replace("https://", f"https://{token}@") if token else REPO
    r = subprocess.run(["git", "clone", "-q", "-b", BRANCH, url, PROJECT], capture_output=True)
    if r.returncode == 0:
        subprocess.run(["git", "-C", PROJECT, "remote", "set-url", "origin", REPO])  # don't keep the token
    else:
        found = [p for d in range(5) for p in glob.glob("/kaggle/input/" + "*/" * d + "src/config.py")]
        if not found:
            raise RuntimeError("git clone failed: add a GITHUB_TOKEN secret or attach the project zip as a dataset")
        print("WARNING: git clone failed; using the attached copy at", os.path.dirname(os.path.dirname(found[0])))
        shutil.copytree(os.path.dirname(os.path.dirname(found[0])), PROJECT,
                        ignore=shutil.ignore_patterns("data", "outputs", "__pycache__"))

shutil.rmtree("/kaggle/working/project", ignore_errors=True)  # remove stale copies saved by older notebook versions
os.chdir(PROJECT)
sys.path.insert(0, PROJECT)
!pip install -q grad-cam anthropic
!git -C {PROJECT} log -1 --format="code version: %h  %s  (%cd)" --date=short 2>/dev/null || echo "code version: attached copy (no git)"

In [ ]:
from src.kaggle_utils import setup
setup(data_dir="/kaggle/working/data")

import glob, os
import pandas as pd
import matplotlib.pyplot as plt
from PIL import Image
from IPython.display import display

## Settings

In [ ]:
FRAMES = 10          # frames per video
MAX_VIDEOS = None    # cap per class (None = all)

metas = glob.glob("/kaggle/input/**/metadata.json", recursive=True)
assert metas, "No metadata.json found: attach DFDC videos (competition data or a dfdc_train_part dataset)"
DFDC_SRC = os.path.commonpath([os.path.dirname(m) for m in metas])
print(f"DFDC root: {DFDC_SRC}  ({len(metas)} metadata.json files)")

In [ ]:
args = f'--src "{DFDC_SRC}" --out /kaggle/working/data --frames {FRAMES} --workers 4'
if MAX_VIDEOS: args += f" --max-videos {MAX_VIDEOS}"
!TQDM_MININTERVAL=60 python -m src.classifier.dfdc_extract {args}

## What was extracted

In [ ]:
m = pd.read_csv("/kaggle/working/data/manifest.csv")
display(m.groupby("label").agg(frames=("file", "count"), videos=("video", "nunique")))
fig, axes = plt.subplots(2, 8, figsize=(17, 4.6))
for row, label in zip(axes, ["real", "fake"]):
    for ax, r in zip(row, m[m.label == label].sample(min(8, (m.label == label).sum()), random_state=0).itertuples()):
        ax.imshow(Image.open(f"/kaggle/working/data/test/{r.label}/{r.file}")); ax.axis("off")
        ax.set_title(f"{label}\n{r.video}", fontsize=8)
plt.tight_layout(); plt.show()

In [ ]:
import shutil
shutil.rmtree("/kaggle/working/outputs", ignore_errors=True)   # keep only the data in this notebook's output
!du -sh /kaggle/working/data